In [ ]:
import numpy as np
import librosa, soundfile as sf
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.signal import iirnotch, filtfilt, butter
import librosa.display


In [ ]:
clean_path = Path("out/100_lead0_16k.wav")
assert clean_path.exists(), "Please generate the ECG .wav file first!"
y_clean, sr = librosa.load(clean_path, sr=None)
t = np.arange(len(y_clean)) / sr

plt.figure(figsize=(12,3))
plt.plot(t[:3*sr], y_clean[:3*sr])
plt.title("Clean ECG Signal (3 seconds)")
plt.xlabel("Time [s]")
plt.ylabel("Amplitude")
plt.show()


In [ ]:
# Simulate biomedical noise
powerline = 0.05 * np.sin(2*np.pi*50*t)        # 50 Hz
baseline  = 0.2 * np.sin(2*np.pi*0.5*t)        # baseline drift
emg       = 0.03 * np.sin(2*np.pi*200*t) + 0.02*np.random.randn(len(y_clean))  # muscle noise

y_noisy = y_clean + powerline + baseline + emg

noisy_path = Path("out/ecg_noisy_realistic.wav")
sf.write(noisy_path, y_noisy.astype(np.float32), sr)
print("✅ Saved:", noisy_path)

plt.figure(figsize=(12,3))
plt.plot(t[:3*sr], y_noisy[:3*sr])
plt.title("Noisy ECG (with baseline + EMG + 50 Hz)")
plt.xlabel("Time [s]")
plt.show()


In [ ]:
def notch_filter(y, sr, notch_freq=50, Q=30):
    w0 = notch_freq / (sr/2)
    b, a = iirnotch(w0, Q)
    return filtfilt(b, a, y)

def highpass_filter(y, sr, cutoff=0.8, order=4):
    nyq = sr / 2
    b, a = butter(order, cutoff/nyq, btype='high')
    return filtfilt(b, a, y)

y_notched = notch_filter(y_noisy, sr)
y_filtered = highpass_filter(y_notched, sr)

sf.write("out/ecg_filtered_clean.wav", y_filtered.astype(np.float32), sr)
print("✅ Saved filtered ECG signal")


In [ ]:
plt.figure(figsize=(12,8))
plt.subplot(3,1,1)
plt.plot(t[:3*sr], y_clean[:3*sr])
plt.title("Clean ECG")

plt.subplot(3,1,2)
plt.plot(t[:3*sr], y_noisy[:3*sr], color='r')
plt.title("Noisy ECG")

plt.subplot(3,1,3)
plt.plot(t[:3*sr], y_filtered[:3*sr], color='g')
plt.title("Filtered ECG (Notch + HPF)")
plt.tight_layout()
plt.show()

# Spectrograms
for sig, title in [(y_noisy, "Noisy ECG"), (y_filtered, "Filtered ECG")]:
    D = librosa.amplitude_to_db(np.abs(librosa.stft(sig, n_fft=2048, hop_length=256)), ref=np.max)
    plt.figure(figsize=(10,3))
    librosa.display.specshow(D, sr=sr, x_axis='time', y_axis='hz', hop_length=256)
    plt.ylim(0,100)
    plt.title(title + " Spectrogram (0–100 Hz)")
    plt.colorbar(format="%+2.0f dB")
    plt.show()


In [ ]:
def snr_db(clean, noisy):
    noise = noisy - clean
    return 10*np.log10(np.sum(clean**2) / np.sum(noise**2))

snr_before = snr_db(y_clean, y_noisy)
snr_after = snr_db(y_clean, y_filtered)

print(f"📊 SNR Before: {snr_before:.2f} dB")
print(f"📊 SNR After : {snr_after:.2f} dB")
print(f"🎯 Improvement: {snr_after - snr_before:.2f} dB")



In [ ]:
from IPython.display import Audio, display
print("🔊 Noisy ECG (for demo only, sounds like low rumble):")
display(Audio(y_noisy, rate=sr))
print("🔊 Filtered ECG (less hum):")
display(Audio(y_filtered, rate=sr))


In [ ]:
import numpy as np, librosa, soundfile as sf, matplotlib.pyplot as plt
from pathlib import Path

# Load your clean ECG
clean_path = Path("out/100_lead0_16k.wav")
y_clean, sr = librosa.load(clean_path, sr=None)
t = np.arange(len(y_clean)) / sr

# Add a STRONG 50Hz hum + random noise
hum = 0.3 * np.sin(2*np.pi*50*t)     # 50Hz mains interference
rand_noise = 0.05 * np.random.randn(len(y_clean))
y_noisy = y_clean + hum + rand_noise

sf.write("out/ecg_with_hum.wav", y_noisy.astype(np.float32), sr)
print("✅ Created: out/ecg_with_hum.wav")

# Plot to show the interference
plt.figure(figsize=(12,3))
plt.plot(y_noisy[:sr*2], color='r')
plt.title("ECG with Clear 50Hz Hum Interference")
plt.show()


In [ ]:
from scipy.signal import iirnotch, filtfilt

def apply_notch(y, sr, notch_freq=50.0, Q=40.0):
    w0 = notch_freq / (sr/2)
    b, a = iirnotch(w0, Q)
    y_filtered = filtfilt(b, a, y)
    return y_filtered

y_filtered = apply_notch(y_noisy, sr)

sf.write("out/ecg_filtered_hum_removed.wav", y_filtered.astype(np.float32), sr)
print("✅ Filtered file saved: out/ecg_filtered_hum_removed.wav")


In [ ]:
import matplotlib.pyplot as plt
from IPython.display import Audio, display

# Play both
print("🔊 Before (Noisy Hum):")
display(Audio(y_noisy, rate=sr))
print("🔊 After (Filtered):")
display(Audio(y_filtered, rate=sr))

# Plot comparison
plt.figure(figsize=(12,5))
plt.subplot(2,1,1)
plt.plot(y_noisy[:sr*2], color='r', alpha=0.7)
plt.title("Before Filtering (Strong 50Hz Hum)")
plt.subplot(2,1,2)
plt.plot(y_filtered[:sr*2], color='g', alpha=0.7)
plt.title("After Notch Filter (50Hz Removed)")
plt.tight_layout()
plt.show()

In [ ]:
import librosa.display
D1 = librosa.amplitude_to_db(np.abs(librosa.stft(y_noisy)), ref=np.max)
D2 = librosa.amplitude_to_db(np.abs(librosa.stft(y_filtered)), ref=np.max)

plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
librosa.display.specshow(D1, sr=sr, x_axis='time', y_axis='hz', cmap='magma')
plt.title("Before Notch Filter")
plt.ylim(0,200)
plt.subplot(1,2,2)
librosa.display.specshow(D2, sr=sr, x_axis='time', y_axis='hz', cmap='magma')
plt.title("After Notch Filter (50Hz Removed)")
plt.ylim(0,200)
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np, librosa, soundfile as sf, matplotlib.pyplot as plt
from pathlib import Path

# Load your clean ECG file
clean_path = Path("out/100_lead0_16k.wav")
y_clean, sr = librosa.load(clean_path, sr=None)
t = np.arange(len(y_clean)) / sr

# --- 1️⃣ Powerline (50 Hz) hum ---
hum = 0.25 * np.sin(2 * np.pi * 50 * t)

# --- 2️⃣ Baseline drift (0.5 Hz respiration-like movement) ---
baseline = 0.4 * np.sin(2 * np.pi * 0.5 * t)

# --- 3️⃣ Muscle / EMG noise (100–300 Hz band + random jitter) ---
emg = 0.05 * np.sin(2 * np.pi * 200 * t) + 0.03 * np.random.randn(len(t))

# Combine everything 🎶
y_noisy = y_clean + hum + baseline + emg

sf.write("out/ecg_noisy_realistic_full.wav", y_noisy.astype(np.float32), sr)
print("✅ Saved: out/ecg_noisy_realistic_full.wav")

# Visualize first few seconds
plt.figure(figsize=(12,4))
plt.plot(t[:3*sr], y_noisy[:3*sr], color='r')
plt.title("ECG with Baseline Drift + 50 Hz Hum + EMG Noise")
plt.xlabel("Time [s]")
plt.ylabel("Amplitude")
plt.show()


In [ ]:
from scipy.signal import iirnotch, butter, filtfilt

def notch_filter(y, sr, notch_freq=50.0, Q=40.0):
    w0 = notch_freq / (sr / 2)
    b, a = iirnotch(w0, Q)
    return filtfilt(b, a, y)

def highpass_filter(y, sr, cutoff=0.8, order=4):
    nyq = sr / 2
    b, a = butter(order, cutoff / nyq, btype='high')
    return filtfilt(b, a, y)

# Apply both
y_notched = notch_filter(y_noisy, sr)
y_filtered = highpass_filter(y_notched, sr)

sf.write("out/ecg_filtered_realistic.wav", y_filtered.astype(np.float32), sr)
print("✅ Saved: out/ecg_filtered_realistic.wav")


In [ ]:
plt.figure(figsize=(12,8))
plt.subplot(3,1,1)
plt.plot(y_clean[:sr*3])
plt.title("Clean ECG")

plt.subplot(3,1,2)
plt.plot(y_noisy[:sr*3], color='r')
plt.title("Noisy ECG (Baseline + 50 Hz + EMG)")

plt.subplot(3,1,3)
plt.plot(y_filtered[:sr*3], color='g')
plt.title("Filtered ECG (Notch + High-pass)")
plt.tight_layout()
plt.show()


In [ ]:
from IPython.display import Audio, display
print("🔊 Before (Noisy ECG):")
display(Audio(y_noisy, rate=sr))
print("🔊 After (Filtered ECG):")
display(Audio(y_filtered, rate=sr))


In [ ]:
import librosa.display
for sig, title in [(y_noisy, "Noisy ECG"), (y_filtered, "Filtered ECG")]:
    D = librosa.amplitude_to_db(np.abs(librosa.stft(sig, n_fft=2048, hop_length=256)), ref=np.max)
    plt.figure(figsize=(10,3))
    librosa.display.specshow(D, sr=sr, x_axis='time', y_axis='hz', hop_length=256, cmap='magma')
    plt.ylim(0,100)
    plt.title(title + " Spectrogram (0–100 Hz)")
    plt.colorbar(format="%+2.0f dB")
    plt.show()


In [ ]:
# ✅ Compute SNR for demo files and save CSV
import numpy as np, soundfile as sf, csv
from pathlib import Path
import math

out_dir = Path("out")
assert out_dir.exists(), "out/ folder not found"

# helper functions
def rms(x): 
    return math.sqrt(float(np.mean(np.asarray(x, dtype=np.float64)**2)) + 1e-12)

def snr_db_from_ref(clean, test):
    # SNR = 10 log10( sum(clean^2) / sum((clean-test)^2) )
    clean = np.asarray(clean, dtype=np.float64)
    test  = np.asarray(test, dtype=np.float64)
    sig = np.sum(clean**2)
    noise = np.sum((clean - test)**2) + 1e-12
    return 10.0 * math.log10((sig) / (noise))

# Strategy:
# - find files with patterns: *_noisy*.wav and matching *_filtered*.wav (or _denoised, _notch, etc.)
# - if a clean reference exists (like combined_clean.wav or same stem without suffix), use it.
pairs = []
wavs = sorted(out_dir.glob("*.wav"))
names = [p.name for p in wavs]
# search heuristics
for p in wavs:
    n = p.stem
    # try find a filtered counterpart: look for same base + keywords
    for key in ("_filtered","_denoised","_notch","_medianSpec","_transient_attenuated","_notch50","_notch60"):
        if n.endswith(key):
            base = n[: -len(key)]
            # find noisy/mixed version with base
            for q in wavs:
                if q.stem.startswith(base) and ("mix" in q.stem or "noisy" in q.stem or "mixed" in q.stem):
                    pairs.append({"clean": None, "noisy": q, "filtered": p})
                    break
    # also check patterns: mixed_X and mixed_X_notched
for q in wavs:
    if "mixed" in q.stem and any(k in q.stem for k in ["mixed","noisy"]):
        # try to find <q>_notched / _denoised / _medianSpec
        for suff in ["_notched","_notch50","_denoised","_medianSpec","_transient_attenuated","_nr","_filtered"]:
            candidate = out_dir / (q.stem + suff + ".wav")
            if candidate.exists():
                pairs.append({"clean": None, "noisy": q, "filtered": candidate})

# If no pairs found, try matching explicit stems (noisy->filtered naming convention)
if not pairs:
    # manual matching by stems: any file that has "noisy" link to file with same stem + "filtered"
    for noisy in wavs:
        if "noisy" in noisy.stem or "mix" in noisy.stem:
            base = noisy.stem.split("_noisy")[0].split("_mix")[0]
            for f in wavs:
                if f.stem.startswith(base) and f != noisy and ("filtered" in f.stem or "denoised" in f.stem or "notch" in f.stem):
                    pairs.append({"clean": None, "noisy": noisy, "filtered": f})

# as fallback, pair explicit files you know (edit these if needed)
fallbacks = [
    {"clean": out_dir / "100_lead0_16k.wav", "noisy": out_dir / "ecg_noisy_realistic_full.wav", "filtered": out_dir / "ecg_filtered_realistic.wav"},
    {"clean": out_dir / "100_lead0_16k.wav", "noisy": out_dir / "ecg_with_hum.wav", "filtered": out_dir / "ecg_filtered_hum_removed.wav"},
]
# add fallback if files exist and not already included
for fb in fallbacks:
    if fb["noisy"].exists() and fb["filtered"].exists():
        already = any(str(fb["noisy"])==str(p["noisy"]) for p in pairs)
        if not already:
            pairs.append(fb)

# remove duplicates
seen = set()
pairs2 = []
for p in pairs:
    key = (str(p['noisy']), str(p['filtered']))
    if key not in seen:
        pairs2.append(p); seen.add(key)
pairs = pairs2

print(f"Found {len(pairs)} noisy/filtered pairs to evaluate.")
rows = []
for idx, pr in enumerate(pairs, start=1):
    noisy_p = Path(pr['noisy'])
    filtered_p = Path(pr['filtered'])
    # try find a clean reference: same stem without suffix or combined_clean.wav
    clean_p = None
    # try common names
    candidates = [out_dir / (noisy_p.stem.split("_mixed")[0] + ".wav"),
                  out_dir / (noisy_p.stem.split("_noisy")[0] + ".wav"),
                  out_dir / "combined_clean.wav",
                  out_dir / "100_lead0_16k.wav"]
    for c in candidates:
        if c.exists():
            clean_p = c
            break
    # load files
    noisy, srn = sf.read(noisy_p)
    filtered, srf = sf.read(filtered_p)
    if srn != srf:
        # resample (simple) -- but here assume same sr
        pass
    # compute RMS
    rms_noisy = rms(noisy)
    rms_filt = rms(filtered)
    if clean_p is not None and clean_p.exists():
        clean_ref, srC = sf.read(clean_p)
        # ensure same length (trim/pad)
        L = min(len(clean_ref), len(filtered))
        snr_before = snr_db_from_ref(clean_ref[:L], noisy[:L])
        snr_after  = snr_db_from_ref(clean_ref[:L], filtered[:L])
    else:
        # fallback estimate: treat filtered as "reduced noise" relative to noisy
        # approximate improvement = 10*log10( sum(filtered^2)/sum((noisy-filtered)^2) ) [not ideal but informative]
        sig = np.sum(filtered.astype(np.float64)**2)
        noise_energy = np.sum((noisy.astype(np.float64)-filtered.astype(np.float64))**2) + 1e-12
        snr_after = 10.0 * math.log10((sig) / (noise_energy))
        # before estimate (no clean) use filtered as proxy for "signal"
        sig_before = np.sum(filtered.astype(np.float64)**2)
        noise_before = np.sum((noisy.astype(np.float64)-filtered.astype(np.float64))**2) + 1e-12
        snr_before = 10.0 * math.log10((sig_before) / (noise_before))
    improvement = snr_after - snr_before
    rows.append({
        "id": idx,
        "noisy": noisy_p.name,
        "filtered": filtered_p.name,
        "clean_ref": clean_p.name if clean_p is not None else "",
        "snr_before_db": round(snr_before, 2),
        "snr_after_db": round(snr_after, 2),
        "improvement_db": round(improvement, 2)
    })

# save CSV
csv_path = out_dir / "results_snr.csv"
with open(csv_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["id","noisy","filtered","clean_ref","snr_before_db","snr_after_db","improvement_db"])
    writer.writeheader()
    for r in rows:
        writer.writerow(r)

print("Saved CSV:", csv_path)
print("Sample results:")
for r in rows[:10]:
    print(r)


In [ ]:
# Cell 1: imports and helpers
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
import soundfile as sf

# Small helpers
def save_wav(path, x, sr):
    # scale to -1..1 float
    x = x.astype(np.float32)
    sf.write(path, x, sr, subtype='FLOAT')

def compute_snr(clean, test):
    L = min(len(clean), len(test))
    clean = clean[:L]
    test = test[:L]
    noise = test - clean
    sig_power = np.sum(clean**2) + 1e-12
    noise_power = np.sum(noise**2) + 1e-12
    return 10 * np.log10(sig_power / noise_power)


In [ ]:
# Cell 2: synthesize ECG-like beats (P-QRS-T as sum of gaussians)
sr = 1000            # sampling rate (Hz) — typical ECG uses 250-1000 Hz
duration = 10.0      # seconds
t = np.arange(0, duration, 1/sr)

hr = 60              # heart rate (bpm) -> beats per second = hr/60
beat_interval = 60.0 / hr

# single beat template as sum of Gaussians (times relative to beat start, seconds)
def make_beat(sr):
    length = int(np.round(beat_interval * sr))
    tb = np.linspace(0, beat_interval, length, endpoint=False)
    beat = np.zeros_like(tb)

    # Parameters: (time, amplitude, width)
    # P wave (small)
    p = (0.16, 0.15, 0.025)
    # Q (small negative)
    q = (0.28, -0.12, 0.01)
    # R (sharp positive)
    r = (0.30, 1.0, 0.008)
    # S (small negative)
    s = (0.33, -0.18, 0.01)
    # T (broader positive)
    t_w = (0.45, 0.25, 0.05)

    comps = [p, q, r, s, t_w]
    for (tc, a, w) in comps:
        beat += a * np.exp(-0.5 * ((tb - tc) / w)**2)

    # scale amplitude to typical arbitrary ECG amplitude (keep relative)
    beat = beat / np.max(np.abs(beat)) * 1.0   # max amplitude = 1.0 (arbitrary units)
    return beat

# tile beats across duration
beat = make_beat(sr)
n_beats = int(np.ceil(duration / beat_interval))
ecg = np.tile(beat, n_beats)[:len(t)]

# Slight smoothing to avoid very sharp edges
ecg = signal.savgol_filter(ecg, 31, 3)  # window 31 samples, polyorder 3

# Optional: normalize to RMS ~0.1 (so noise levels are meaningful)
rms_target = 0.1
ecg = ecg / (np.sqrt(np.mean(ecg**2)) + 1e-12) * rms_target

print("Synthesized ECG: sr =", sr, "duration =", duration, "s")


In [ ]:
# Cell 3: add noises
np.random.seed(0)

# 50 Hz mains hum (sinusoid)
hum_freq = 50.0
hum_amp = 0.12  # relative to ECG RMS, adjust to taste
hum = hum_amp * np.sin(2*np.pi*hum_freq*t)

# Baseline wander (very low freq)
bw_freq = 0.3   # 0.3 Hz baseline drift
bw_amp = 0.2
baseline_wander = bw_amp * np.sin(2*np.pi*bw_freq*t)

# EMG-like high-frequency noise (band-limited)
emg = 0.06 * signal.lfilter([1.0], [1.0], np.random.randn(len(t)))  # small broad noise

# White Gaussian noise
wgn = 0.03 * np.random.randn(len(t))

# Combine
noisy_ecg = ecg + hum + baseline_wander + emg + wgn

# Save clean / noisy audio files (optional)
save_wav("ecg_clean.wav", ecg, sr)
save_wav("ecg_noisy.wav", noisy_ecg, sr)

print("Saved: ecg_clean.wav, ecg_noisy.wav")


In [ ]:
# Cell 4: plots to inspect
import matplotlib.pyplot as plt

seg = int(sr * 3)  # show first 3 seconds
time_seg = t[:seg]

plt.figure(figsize=(10,3))
plt.plot(time_seg, ecg[:seg], label='clean')
plt.plot(time_seg, noisy_ecg[:seg], alpha=0.6, label='noisy')
plt.legend()
plt.xlabel("Time (s)")
plt.title("Waveform (first 3 s)")
plt.tight_layout()
plt.show()

# Spectrogram of noisy
plt.figure(figsize=(10,3))
plt.specgram(noisy_ecg, NFFT=1024, Fs=sr, noverlap=512)
plt.title("Spectrogram - noisy (look for 50 Hz line)")
plt.xlabel("Time (s)")
plt.ylabel("Frequency (Hz)")
plt.colorbar(label='dB')
plt.tight_layout()
plt.show()

# PSD (Welch) to highlight 50 Hz
f, Pxx = signal.welch(noisy_ecg, sr, nperseg=4096)
plt.figure(figsize=(8,3))
plt.semilogy(f, Pxx)
plt.xlim(0,200)
plt.axvline(50, color='r', linestyle='--', label='50 Hz')
plt.title("PSD (Welch) - noisy")
plt.xlabel("Frequency (Hz)")
plt.ylabel("Power")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Cell 5: Notch filter at 50 Hz (IIR notch) + optional bandpass 0.5-40 Hz
fs = sr
f0 = 50.0
Q = 30.0  # quality factor (higher -> narrower notch)
b_notch, a_notch = signal.iirnotch(f0, Q, fs)

# zero-phase apply
noisy_notched = signal.filtfilt(b_notch, a_notch, noisy_ecg)

# Optional bandpass 0.5-40 Hz (4th order Butterworth) to preserve ECG band
lowcut = 0.5
highcut = 40.0
b_bp, a_bp = signal.butter(4, [lowcut/(fs/2), highcut/(fs/2)], btype='band')
noisy_notched_bp = signal.filtfilt(b_bp, a_bp, noisy_notched)

# Save outputs
save_wav("ecg_filtered_notch.wav", noisy_notched, sr)
save_wav("ecg_filtered_notch_bandpass.wav", noisy_notched_bp, sr)

print("Saved: ecg_filtered_notch.wav, ecg_filtered_notch_bandpass.wav")


In [ ]:
# Cell 6: SNR & visual comparisons
snr_before = compute_snr(ecg, noisy_ecg)
snr_after_notch = compute_snr(ecg, noisy_notched)
snr_after_notch_bp = compute_snr(ecg, noisy_notched_bp)
print(f"SNR (clean vs noisy): {snr_before:.2f} dB")
print(f"SNR (clean vs after notch): {snr_after_notch:.2f} dB")
print(f"SNR (clean vs after notch+bandpass): {snr_after_notch_bp:.2f} dB")
print(f"SNR improvement (notch): {snr_after_notch - snr_before:.2f} dB")

# Waveform compare
plt.figure(figsize=(10,3))
plt.plot(time_seg, noisy_ecg[:seg], alpha=0.5, label='noisy')
plt.plot(time_seg, noisy_notched[:seg], label='after notch')
plt.plot(time_seg, noisy_notched_bp[:seg], label='notch + bandpass')
plt.plot(time_seg, ecg[:seg], '--', label='clean (reference)', alpha=0.9)
plt.legend()
plt.title("Waveform comparison (first 3 s)")
plt.xlabel("Time (s)")
plt.tight_layout()
plt.show()

# Spectrogram after filtering
plt.figure(figsize=(10,3))
plt.specgram(noisy_notched_bp, NFFT=1024, Fs=sr, noverlap=512)
plt.title("Spectrogram - after notch + bandpass")
plt.xlabel("Time (s)")
plt.ylabel("Frequency (Hz)")
plt.colorbar()
plt.tight_layout()
plt.show()

# PSD compare
f, P_noisy = signal.welch(noisy_ecg, fs, nperseg=4096)
f, P_filtered = signal.welch(noisy_notched_bp, fs, nperseg=4096)
plt.figure(figsize=(8,3))
plt.semilogy(f, P_noisy, label='noisy')
plt.semilogy(f, P_filtered, label='filtered')
plt.xlim(0,200)
plt.axvline(50, color='r', linestyle='--', label='50 Hz')
plt.legend()
plt.title("PSD before vs after")
plt.tight_layout()
plt.show()


In [ ]:
import os
print("Files in cwd:", os.listdir())
print("Sizes:")
for f in ["ytstyle_ecg_noisy_pcm16.wav","ytstyle_ecg_filtered_pcm16.wav"]:
    if os.path.exists(f):
        print(f, os.path.getsize(f))
    else:
        print(f, "MISSING")


In [ ]:
pip install wfdb


In [ ]:
# Cell 1: search your data_ecg folder for .hea/.dat files and list available record basenames
import os, glob, textwrap

base_path = r"C:\notch_filter_pocs\notch_project\data_ecg\mit-bih-arrhythmia-database-1.0.0"
print("Searching:", base_path)

# find .hea and .dat files (recursively)
hea_files = glob.glob(os.path.join(base_path, "**", "*.hea"), recursive=True)
dat_files = glob.glob(os.path.join(base_path, "**", "*.dat"), recursive=True)

print("\nFound .hea files (first 20):")
for i,f in enumerate(hea_files[:20],1):
    print(f" {i}. {os.path.relpath(f, base_path)}")

print("\nFound .dat files (first 20):")
for i,f in enumerate(dat_files[:20],1):
    print(f" {i}. {os.path.relpath(f, base_path)}")

if not hea_files and not dat_files:
    print(textwrap.dedent("""
    No .hea/.dat files were found under the path.
    Possible reasons:
     - The dataset archive wasn't extracted (you may have a zip file instead).
     - Files are in a different folder.
    If you see a zip file like mit-bih-arrhythmia-database-1.0.0.zip in the parent, unzip it and re-run this cell.
    """))
else:
    # build list of candidate record basenames (strip extensions and any record suffix)
    records = sorted({ os.path.splitext(os.path.basename(f))[0] for f in hea_files + dat_files })
    print("\nCandidate record basenames (examples):", records[:10])
    # store for next cell
    print("\nIf you want to load record X, use its basename from the list above (e.g., '100').")


In [ ]:
# Robust loader for local WFDB records on Windows
import os, glob, wfdb
import soundfile as sf
from IPython.display import Audio, display
import numpy as np
import traceback

base_path = r"C:\notch_filter_pocs\notch_project\data_ecg\mit-bih-arrhythmia-database-1.0.0"
print("Base path:", base_path)
print("Exists?", os.path.exists(base_path))

# Find files
hea_files = glob.glob(os.path.join(base_path, "**", "*.hea"), recursive=True)
dat_files = glob.glob(os.path.join(base_path, "**", "*.dat"), recursive=True)
print("Found hea count:", len(hea_files), "  dat count:", len(dat_files))

if not hea_files and not dat_files:
    print("No .hea/.dat files found under base_path. Show directory listing of base path:")
    for f in os.listdir(base_path):
        print("  ", f)
    raise FileNotFoundError("No WFDB files under base_path. If you have a zip, please extract it.")

# pick first hea if available otherwise dat
if hea_files:
    chosen = hea_files[0]
else:
    chosen = dat_files[0]
print("Chosen file (for discovery):", chosen)
record_dir = os.path.dirname(chosen)
record_basename = os.path.splitext(os.path.basename(chosen))[0]
print("Record folder:", record_dir)
print("Record basename:", record_basename)

# Show files right there to confirm exact names
print("\nFiles in record folder (sample):")
for f in os.listdir(record_dir)[:200]:
    print("  ", f)

# Try a few ways to load. We'll stop at first success.
success = False
errors = []

# 1) Try wfdb.rdsamp with full path (works well on Windows)
try:
    print("\nTrying wfdb.rdsamp with full path (no pn_dir)...")
    rec_path = os.path.join(record_dir, record_basename)
    # replace backslashes with forward slashes for wfdb (sometimes helps)
    rec_path_posix = rec_path.replace("\\", "/")
    sig, fields = wfdb.rdsamp(rec_path_posix)
    fs = fields.get('fs', None) or fields.get('freq', None) or getattr(fields, 'fs', None)
    if fs is None and 'fs' in fields:
        fs = fields['fs']
    if fs is None:
        fs = 360  # fallback guess
    print("rdsamp loaded, fs =", fs, " shape:", sig.shape)
    sig0 = sig[:,0] if sig.ndim>1 else sig
    success = True
except Exception as e:
    errors.append(("rdsamp", str(e)))
    print("rdsamp failed:", e)

# 2) Try wfdb.rdrecord with pn_dir using posix-style folder
if not success:
    try:
        print("\nTrying wfdb.rdrecord with pn_dir (posix-style folder)...")
        rec_name = record_basename
        pn_dir = record_dir.replace("\\", "/")
        rec = wfdb.rdrecord(rec_name, pn_dir=pn_dir)
        fs = rec.fs
        sig0 = rec.p_signal[:,0]
        print("rdrecord loaded, fs =", fs, " len sig0 =", len(sig0))
        success = True
    except Exception as e:
        errors.append(("rdrecord", str(e)))
        print("rdrecord failed:", e)

# 3) Try wfdb.rdrecord with absolute path given as record_name (without pn_dir)
if not success:
    try:
        print("\nTrying wfdb.rdrecord with absolute path as record_name (no pn_dir)...")
        rec_path_noslash = rec_path  # already has backslashes
        rec2 = wfdb.rdrecord(rec_path_noslash)
        fs = rec2.fs
        sig0 = rec2.p_signal[:,0]
        print("rdrecord-with-fullpath loaded, fs =", fs, " len sig0 =", len(sig0))
        success = True
    except Exception as e:
        errors.append(("rdrecord-fullpath", str(e)))
        print("rdrecord-fullpath failed:", e)

# 4) As a last resort, try reading header text to inspect it
if not success:
    try:
        print("\nTrying to read header file manually to inspect (for debugging)...")
        header_path = os.path.join(record_dir, record_basename + ".hea")
        with open(header_path, "r", encoding="ascii", errors="ignore") as f:
            header_txt = f.read()
        print("Header file preview (first 400 chars):")
        print(header_txt[:400])
    except Exception as e:
        errors.append(("read-header", str(e)))
        print("Failed reading header file:", e)

# If we succeeded, write a 10 s wav and play
if success:
    try:
        # ensure sig0 is numpy array and float
        sig0 = np.array(sig0, dtype=np.float64)
        if len(sig0) < 1:
            raise ValueError("Loaded signal is empty.")
        # pick a 10 second window (or shorter if file shorter)
        if 'fs' not in locals() or fs is None:
            fs = 360
        seg_len = min(len(sig0), int(10 * fs))
        seg = sig0[:seg_len]
        # normalize to avoid clipping and scale to audible range
        seg = seg / (np.max(np.abs(seg)) + 1e-12) * 0.6
        out_fn = os.path.join(record_dir, record_basename + "_clean_10s_pcm16.wav")
        sf.write(out_fn, seg, int(fs), subtype='PCM_16')
        print("\nWrote clean WAV:", out_fn, " duration:", len(seg)/fs)
        display(Audio(out_fn))
    except Exception as e:
        print("Failed to write/play WAV:", e)
        traceback.print_exc()
else:
    print("\nAll load attempts failed. Errors summary:")
    for tag, msg in errors:
        print(f" - {tag}: {msg}")
    print("\nPlease paste the file listing printed above (the 'Files in record folder') so I can pick the correct filenames and give an exact load command.")


In [ ]:
import os, soundfile as sf
fn = r"C:\notch_filter_pocs\notch_project\data_ecg\mit-bih-arrhythmia-database-1.0.0\mit-bih-arrhythmia-database-1.0.0\100_clean_10s_pcm16.wav"
print("Exists:", os.path.exists(fn))
print("Size (bytes):", os.path.getsize(fn))
info = sf.info(fn)
print("SoundFile info:", info)
x, sr = sf.read(fn)
print("Loaded samples:", len(x), " duration (s):", len(x)/sr, " sr:", sr, " dtype:", x.dtype, " min/max:", x.min(), x.max())


In [ ]:
# SINGLE CELL: produce clean / noisy / filtered WAVs from MIT-BIH record 100,
# compute SNR, plot waveform + spectrogram + PSD, and write a single demo-trio audio.
# Paste & run in your notebook (you already loaded wfdb earlier).
import os, glob, wfdb
import numpy as np
from scipy import signal
import soundfile as sf
import matplotlib.pyplot as plt
from IPython.display import Audio, display

# -------------------- USER SETTINGS --------------------
record_name = "100"   # change if you want a different record
# base folder where the MIT-BIH folder lives (change if different)
root_folder = r"C:\notch_filter_pocs\notch_project\data_ecg\mit-bih-arrhythmia-database-1.0.0\mit-bih-arrhythmia-database-1.0.0"
# where final files will be saved
out_folder = root_folder
# choose start second for the 10s segment (0..)
start_second = 0
duration_s = 10
# noise knobs — tweak to match the YouTube short
buzz_rms = 0.18   # energy of mains-style buzz
hum_harmonics = [1,2,3,4,5]
am_depth = 0.55
am_rate = 2.0
click_count = 4
baseline_amp = 0.18
hf_grit = 0.12
# notch filter params
notch_f0 = 50.0
notch_Q = 30.0
# bandpass to preserve ECG morphology
bp_low = 0.5
bp_high = 45.0
# -------------------------------------------------------

# --- load record (robust) ---
record_path = os.path.join(root_folder, f"{record_name}.hea")
if not os.path.exists(record_path):
    raise FileNotFoundError(f"Header not found: {record_path}\nRun the discovery cell if path differs.")

rec = wfdb.rdrecord(os.path.join(root_folder, record_name))
fs = int(rec.fs)
sig = rec.p_signal  # shape (N, channels)
if sig.ndim>1:
    ecg_full = sig[:,0].astype(np.float64)
else:
    ecg_full = sig.astype(np.float64)

# pick segment
s0 = int(start_second * fs)
seg_len = min(len(ecg_full) - s0, int(duration_s * fs))
if seg_len <= 0:
    raise ValueError("Segment length <=0; adjust start_second or duration.")
segment = ecg_full[s0:s0+seg_len]
# normalize & scale to audible/amplitude-safe range
segment = segment / (np.max(np.abs(segment)) + 1e-12) * 0.6
sr = fs

# --- build YouTube-style noisy version ---
t = np.arange(len(segment)) / sr
np.random.seed(5)

# buzz: harmonic series of mains with small jitter + AM
buzz = np.zeros_like(t)
jitter = 0.08 * np.sin(2*np.pi*0.13*t + 0.3)
for h in hum_harmonics:
    amp = (0.35 / h)
    freq = 50.0 * h * (1.0 + 0.0006 * np.random.randn())
    phase = 2*np.pi*(freq * t + jitter * h)
    buzz += amp * np.sin(phase)
# amplitude modulation (gives wobble)
am = 1.0 - am_depth * (0.5*(1 + np.sin(2*np.pi*am_rate*t)))
buzz *= am
# add HF grit (band-limited noise)
hf_noise = 0.08 * np.random.randn(len(t))
# design a small lowpass to shape that noise (keeping energy under ~200Hz for ECG sr)
b_lp = signal.firwin(201, min(200.0/(sr/2), 0.99))
hf_noise = signal.lfilter(b_lp, [1.0], hf_noise)
buzz += hf_grit * hf_noise
# scale buzz to desired rms
buzz = buzz / (np.sqrt(np.mean(buzz**2)) + 1e-12) * buzz_rms

# clicks (random transients)
clicks = np.zeros_like(t)
click_positions = np.linspace(1.5, duration_s-1.5, click_count) + 0.25*np.random.randn(click_count)
for cp in click_positions:
    idx = int(cp * sr)
    w = int(0.004 * sr)
    if 0 <= idx < len(t) and idx + w < len(t):
        tt = np.arange(w)/sr
        clicks[idx:idx+w] += 0.6 * np.exp(-((tt-0.002)/0.001)**2) * (np.random.rand()*0.6 + 0.4)

# baseline wander
baseline = baseline_amp * np.sin(2*np.pi*0.25*t + 0.4)

# Combine noisy signal
noisy = segment + buzz + clicks + baseline + 0.02*np.random.randn(len(t))
# normalize to avoid clipping
mx = np.max(np.abs(noisy)) + 1e-12
if mx > 0.98:
    noisy = noisy / mx * 0.98

# --- apply notch + bandpass (zero-phase) ---
b_notch, a_notch = signal.iirnotch(notch_f0, notch_Q, sr)
noisy_notched = signal.filtfilt(b_notch, a_notch, noisy)
# bandpass
b_bp, a_bp = signal.butter(4, [bp_low/(sr/2), bp_high/(sr/2)], btype='band')
filtered = signal.filtfilt(b_bp, a_bp, noisy_notched)

# --- write PCM16 WAVs ---
def write_pcm16(path, x, sr):
    x_out = np.array(x, dtype=np.float32)
    m = np.max(np.abs(x_out)) + 1e-12
    if m > 1.0:
        x_out = x_out / m * 0.98
    sf.write(path, x_out, sr, subtype='PCM_16')
    return os.path.getsize(path)

clean_fn = os.path.join(out_folder, f"{record_name}_clean_10s_pcm16.wav")
noisy_fn = os.path.join(out_folder, f"{record_name}_ytstyle_noisy_10s_pcm16.wav")
filt_fn  = os.path.join(out_folder, f"{record_name}_ytstyle_filtered_10s_pcm16.wav")
write_pcm16(clean_fn, segment, sr)
write_pcm16(noisy_fn, noisy, sr)
write_pcm16(filt_fn, filtered, sr)

# --- compute SNRs (clean reference is 'segment') ---
def compute_snr(clean, test):
    L = min(len(clean), len(test))
    clean = clean[:L]
    test = test[:L]
    noise = test - clean
    sig_power = np.sum(clean**2) + 1e-12
    noise_power = np.sum(noise**2) + 1e-12
    return 10*np.log10(sig_power / noise_power)

snr_noisy = compute_snr(segment, noisy)
snr_filtered = compute_snr(segment, filtered)

# --- make a single demo file with 1s silence between clips ---
sil = np.zeros(int(1.0*sr))
demo = np.concatenate([segment, sil, noisy, sil, filtered])
demo_fn = os.path.join(out_folder, f"{record_name}_demo_trio.wav")
write_pcm16(demo_fn, demo, sr)

# --- print results & show players ---
print("Saved files to:", out_folder)
print("  CLEAN:", os.path.basename(clean_fn))
print("  NOISY:", os.path.basename(noisy_fn))
print("  FILTERED:", os.path.basename(filt_fn))
print("  DEMO (clean→noisy→filtered):", os.path.basename(demo_fn))
print()
print(f"SNR (clean vs noisy): {snr_noisy:.2f} dB")
print(f"SNR (clean vs filtered): {snr_filtered:.2f} dB")
print()

# --- quick plots: waveform (first 4s), spectrogram noisy vs filtered, PSD ---
plt.figure(figsize=(10,3))
tseg = t[:min(len(t), 4*sr)]
plt.plot(tseg, segment[:len(tseg)], label='clean')
plt.plot(tseg, noisy[:len(tseg)], alpha=0.6, label='noisy')
plt.plot(tseg, filtered[:len(tseg)], label='filtered', alpha=0.9)
plt.legend(); plt.title('Waveform (first 4 s)'); plt.xlabel('Time (s)')
plt.tight_layout(); plt.show()

plt.figure(figsize=(10,3))
plt.specgram(noisy, NFFT=1024, Fs=sr, noverlap=512)
plt.title('Spectrogram - noisy'); plt.xlabel('Time (s)'); plt.ylabel('Freq (Hz)')
plt.colorbar(); plt.tight_layout(); plt.show()

plt.figure(figsize=(10,3))
plt.specgram(filtered, NFFT=1024, Fs=sr, noverlap=512)
plt.title('Spectrogram - filtered'); plt.xlabel('Time (s)'); plt.ylabel('Freq (Hz)')
plt.colorbar(); plt.tight_layout(); plt.show()

f, Pn = signal.welch(noisy, sr, nperseg=4096)
f, Pf = signal.welch(filtered, sr, nperseg=4096)
plt.figure(figsize=(8,3))
plt.semilogy(f, Pn, label='noisy')
plt.semilogy(f, Pf, label='filtered')
plt.xlim(0,200); plt.axvline(50, color='r', linestyle='--', label='50 Hz')
plt.legend(); plt.title('PSD before vs after'); plt.tight_layout(); plt.show()

# --- display small audio players (guaranteed method) ---
print("Players (clean, noisy, filtered, demo):")
display(Audio(segment, rate=sr))
display(Audio(noisy, rate=sr))
display(Audio(filtered, rate=sr))
display(Audio(demo, rate=sr))

# Optionally open in system media player (Windows)
try:
    os.startfile(demo_fn)
    print("Opened demo file in default system player:", demo_fn)
except Exception:
    pass


In [ ]:
# Recheck, repair, and verify audio signals (clean/noisy/filtered) and confirm playback.
import soundfile as sf, numpy as np, os
import matplotlib.pyplot as plt
from scipy import signal
from IPython.display import Audio, display

# Paths (update if you used diff folder)
base = r"C:\notch_filter_pocs\notch_project\data_ecg\mit-bih-arrhythmia-database-1.0.0\mit-bih-arrhythmia-database-1.0.0"
clean_fn = os.path.join(base, "100_clean_10s_pcm16.wav")
noisy_fn = os.path.join(base, "100_ytstyle_noisy_10s_pcm16.wav")
filt_fn  = os.path.join(base, "100_ytstyle_filtered_10s_pcm16.wav")
demo_fn  = os.path.join(base, "100_demo_trio.wav")

files = [clean_fn, noisy_fn, filt_fn, demo_fn]
for f in files:
    print("\n---", os.path.basename(f), "---")
    if not os.path.exists(f):
        print("❌ File missing.")
        continue
    info = sf.info(f)
    x, sr = sf.read(f, always_2d=False)
    print("✓ Exists | sr:", info.samplerate, "| channels:", info.channels, "| len samples:", len(x), "| dur(s):", len(x)/sr)
    if np.allclose(x, 0, atol=1e-9):
        print("⚠️ Silent file (all zeros).")
    else:
        print("Signal stats: min {:.4f}, max {:.4f}, mean {:.4f}".format(np.min(x), np.max(x), np.mean(x)))

# Fix if filtered file silent or corrupted
if os.path.exists(filt_fn):
    x_f, sr = sf.read(filt_fn)
    if np.allclose(x_f, 0, atol=1e-9) or len(x_f) == 0:
        print("\nRecreating filtered from noisy (just in case).")
        x_n, sr = sf.read(noisy_fn)
        b_notch, a_notch = signal.iirnotch(50.0, 30.0, sr)
        b_bp, a_bp = signal.butter(4, [0.5/(sr/2), 45.0/(sr/2)], btype='band')
        x_nf = signal.filtfilt(b_notch, a_notch, x_n)
        x_filt = signal.filtfilt(b_bp, a_bp, x_nf)
        x_filt = x_filt / (np.max(np.abs(x_filt)) + 1e-12) * 0.9
        sf.write(filt_fn, x_filt, sr, subtype='PCM_16')
        print("✅ Filtered file recreated:", filt_fn)
        display(Audio(x_filt, rate=sr))
    else:
        print("\nFiltered file OK; showing preview:")
        display(Audio(x_f, rate=sr))

# Plot all three
plt.figure(figsize=(10,6))
for fn,color,label in zip([clean_fn,noisy_fn,filt_fn],['g','r','b'],['Clean','Noisy','Filtered']):
    if os.path.exists(fn):
        x, sr = sf.read(fn)
        t = np.arange(len(x))/sr
        plt.plot(t, x, color=color, alpha=0.7, label=label)
plt.legend(); plt.xlabel("Time (s)"); plt.ylabel("Amplitude")
plt.title("Waveforms: Clean vs Noisy vs Filtered (10 s)")
plt.tight_layout(); plt.show()

# Spectrum comparison
plt.figure(figsize=(10,4))
for fn,label in zip([noisy_fn,filt_fn],['Noisy','Filtered']):
    if os.path.exists(fn):
        x, sr = sf.read(fn)
        f, Pxx = signal.welch(x, sr, nperseg=min(2048,len(x)//2))
        plt.semilogy(f, Pxx, label=label)
plt.axvline(50, color='r', linestyle='--', label='50 Hz')
plt.xlim(0,200); plt.legend(); plt.xlabel("Freq (Hz)"); plt.ylabel("PSD")
plt.title("PSD: 50 Hz notch removal check")
plt.tight_layout(); plt.show()


In [ ]:
import os
base = r"C:\notch_filter_pocs\notch_project\data_ecg\mit-bih-arrhythmia-database-1.0.0\mit-bih-arrhythmia-database-1.0.0"
for f in os.listdir(base):
    if f.endswith(".wav"):
        print(f, os.path.getsize(os.path.join(base,f)))


In [ ]:
import os
fn = r"C:\notch_filter_pocs\notch_project\data_ecg\mit-bih-arrhythmia-database-1.0.0\mit-bih-arrhythmia-database-1.0.0\100_demo_trio.wav"
os.startfile(fn)


In [ ]:
import os
base = r"C:\notch_filter_pocs\notch_project\data_ecg\mit-bih-arrhythmia-database-1.0.0\mit-bih-arrhythmia-database-1.0.0"
for f in os.listdir(base):
    if f.endswith(".wav"):
        print(f)


In [ ]:
import os
os.startfile(r"C:\notch_filter_pocs\notch_project\data_ecg\mit-bih-arrhythmia-database-1.0.0\mit-bih-arrhythmia-database-1.0.0\100_demo_trio.wav")


In [ ]:
import os
base = r"C:\notch_filter_pocs\notch_project\data_ecg\mit-bih-arrhythmia-database-1.0.0\mit-bih-arrhythmia-database-1.0.0"

os.startfile(os.path.join(base, "100_clean_10s_pcm16.wav"))      # clean ECG
os.startfile(os.path.join(base, "100_ytstyle_noisy_10s_pcm16.wav"))   # noisy ECG
os.startfile(os.path.join(base, "100_ytstyle_filtered_10s_pcm16.wav")) # filtered ECG


In [ ]:
import os
demo = r"C:\notch_filter_pocs\notch_project\data_ecg\mit-bih-arrhythmia-database-1.0.0\mit-bih-arrhythmia-database-1.0.0\100_demo_trio.wav"
os.startfile(demo)


In [ ]:
import soundfile as sf, numpy as np, os

base = r"C:\notch_filter_pocs\notch_project\data_ecg\mit-bih-arrhythmia-database-1.0.0\mit-bih-arrhythmia-database-1.0.0"
clean = os.path.join(base,"100_clean_10s_pcm16.wav")
noisy = os.path.join(base,"100_ytstyle_noisy_10s_pcm16.wav")
filt  = os.path.join(base,"100_ytstyle_filtered_10s_pcm16.wav")
out_demo = os.path.join(base,"100_demo_new.wav")

# read
x1, sr1 = sf.read(clean)
x2, sr2 = sf.read(noisy)
x3, sr3 = sf.read(filt)
assert sr1 == sr2 == sr3
sr = sr1

# 1 second silence
sil = np.zeros(int(1*sr))

demo = np.concatenate([x1, sil, x2, sil, x3])
# normalize a bit to avoid clipping
demo = demo / (np.max(np.abs(demo)) + 1e-12) * 0.95

sf.write(out_demo, demo.astype(np.float32), sr, subtype='PCM_16')
print("Wrote:", out_demo)
os.startfile(out_demo)   # open in default system player


In [ ]:
import os
base = r"C:\notch_filter_pocs\notch_project\data_ecg\mit-bih-arrhythmia-database-1.0.0\mit-bih-arrhythmia-database-1.0.0"
for fn in ["100_clean_10s_pcm16.wav","100_ytstyle_noisy_10s_pcm16.wav","100_ytstyle_filtered_10s_pcm16.wav","100_demo_trio.wav"]:
    p = os.path.join(base, fn)
    print(fn, "->", "EXISTS" if os.path.exists(p) else "MISSING", "| size (bytes):", os.path.getsize(p) if os.path.exists(p) else "-")


In [ ]:
import os
demo = r"C:\notch_filter_pocs\notch_project\data_ecg\mit-bih-arrhythmia-database-1.0.0\mit-bih-arrhythmia-database-1.0.0\100_demo_trio.wav"
print("Opening:", demo)
os.startfile(demo)


In [ ]:
import os

base = r"C:\notch_filter_pocs\notch_project\data_ecg\mit-bih-arrhythmia-database-1.0.0\mit-bih-arrhythmia-database-1.0.0"
files = [
    ("Clean ECG (original)", "100_clean_10s_pcm16_sr16000.wav"),
    ("Noisy ECG (with hum + interference)", "100_ytstyle_noisy_10s_pcm16_sr16000.wav"),
    ("Filtered ECG (after notch + bandpass)", "100_ytstyle_filtered_10s_pcm16_sr16000.wav"),
    ("Demo (clean→noisy→filtered)", "100_demo_trio_sr16000.wav")
]

for title, fn in files:
    path = os.path.join(base, fn)
    print(f"▶️  {title}")
    os.startfile(path)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import wfdb
import os
from scipy import signal
import soundfile as sf


In [ ]:
base_path = r"C:\notch_filter_pocs\notch_project\data_ecg\mit-bih-arrhythmia-database-1.0.0\mit-bih-arrhythmia-database-1.0.0"

record = wfdb.rdrecord(os.path.join(base_path, "100"))
ecg = record.p_signal[:, 0]  # first channel
fs = int(record.fs)
print(f"Loaded ECG: {len(ecg)} samples, Sampling rate = {fs} Hz")


In [ ]:
duration = len(ecg) / fs
t = np.linspace(0, duration, len(ecg))

# Add 50 Hz hum and random EMG-like noise
hum = 0.2 * np.sin(2 * np.pi * 50 * t)
emg_noise = 0.05 * np.random.randn(len(ecg))
noisy_ecg = ecg + hum + emg_noise

print("✅ Noise added successfully.")


In [ ]:
# Notch filter (50 Hz)
f0 = 50.0
Q = 30.0
b_notch, a_notch = signal.iirnotch(f0, Q, fs)
ecg_notched = signal.filtfilt(b_notch, a_notch, noisy_ecg)

# Bandpass filter (0.5–45 Hz)
lowcut, highcut = 0.5, 45.0
b_band, a_band = signal.butter(4, [lowcut/(fs/2), highcut/(fs/2)], btype='band')
ecg_filtered = signal.filtfilt(b_band, a_band, ecg_notched)

print("✅ Applied notch + bandpass filtering.")


In [ ]:
plt.figure(figsize=(12,6))
plt.subplot(3,1,1)
plt.plot(ecg[:1500], color='green')
plt.title("Clean ECG (Original)")
plt.subplot(3,1,2)
plt.plot(noisy_ecg[:1500], color='red')
plt.title("Noisy ECG (with hum + EMG noise)")
plt.subplot(3,1,3)
plt.plot(ecg_filtered[:1500], color='blue')
plt.title("Filtered ECG (after notch + bandpass)")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(10,5))
f_clean, Pxx_clean = signal.welch(ecg, fs, nperseg=2048)
f_noisy, Pxx_noisy = signal.welch(noisy_ecg, fs, nperseg=2048)
f_filt, Pxx_filt = signal.welch(ecg_filtered, fs, nperseg=2048)

plt.semilogy(f_noisy, Pxx_noisy, label='Noisy ECG', color='red')
plt.semilogy(f_filt, Pxx_filt, label='Filtered ECG', color='blue')
plt.xlim(0, 100)
plt.xlabel("Frequency (Hz)")
plt.ylabel("Power Spectral Density")
plt.title("50 Hz Peak Removed After Filtering")
plt.legend()
plt.show()


In [ ]:
# 🎨 Spectrogram Comparison for Clean, Noisy, and Filtered ECG
from scipy import signal
import matplotlib.pyplot as plt

plt.figure(figsize=(12,8))

# Clean ECG
plt.subplot(3,1,1)
f, t_spec, Sxx = signal.spectrogram(ecg[:fs*5], fs, nperseg=512, noverlap=256)
plt.pcolormesh(t_spec, f, 10*np.log10(Sxx + 1e-10), shading='gouraud', cmap='viridis')
plt.title("Spectrogram - Clean ECG")
plt.ylabel("Frequency (Hz)")
plt.colorbar(label="Power (dB)")

# Noisy ECG
plt.subplot(3,1,2)
f, t_spec, Sxx = signal.spectrogram(noisy_ecg[:fs*5], fs, nperseg=512, noverlap=256)
plt.pcolormesh(t_spec, f, 10*np.log10(Sxx + 1e-10), shading='gouraud', cmap='viridis')
plt.title("Spectrogram - Noisy ECG (with 50Hz hum)")
plt.ylabel("Frequency (Hz)")
plt.colorbar(label="Power (dB)")

# Filtered ECG
plt.subplot(3,1,3)
f, t_spec, Sxx = signal.spectrogram(ecg_filtered[:fs*5], fs, nperseg=512, noverlap=256)
plt.pcolormesh(t_spec, f, 10*np.log10(Sxx + 1e-10), shading='gouraud', cmap='viridis')
plt.title("Spectrogram - Filtered ECG (after notch + bandpass)")
plt.xlabel("Time (s)")
plt.ylabel("Frequency (Hz)")
plt.colorbar(label="Power (dB)")

plt.tight_layout()
plt.show()


In [ ]:
# ⚡ Power Spectral Density (PSD) Comparison
from scipy import signal
import matplotlib.pyplot as plt

# Compute PSD using Welch’s method
f_clean, Pxx_clean = signal.welch(ecg[:fs*5], fs, nperseg=2048)
f_noisy, Pxx_noisy = signal.welch(noisy_ecg[:fs*5], fs, nperseg=2048)
f_filt, Pxx_filt = signal.welch(ecg_filtered[:fs*5], fs, nperseg=2048)

# Plot
plt.figure(figsize=(10,6))
plt.semilogy(f_clean, Pxx_clean, label='Clean ECG', color='green')
plt.semilogy(f_noisy, Pxx_noisy, label='Noisy ECG', color='red')
plt.semilogy(f_filt, Pxx_filt, label='Filtered ECG', color='blue')

plt.xlim(0, 100)
plt.xlabel("Frequency (Hz)")
plt.ylabel("Power Spectral Density")
plt.title("PSD Comparison — 50 Hz Peak Removed After Filtering")
plt.legend()
plt.grid(True, which='both', linestyle='--', linewidth=0.5)
plt.show()
plt.axvline(50, color='black', linestyle='--', linewidth=1)
plt.text(51, max(Pxx_noisy)/5, '50 Hz hum removed →', color='black')


In [ ]:
sf.write(os.path.join(base_path, "100_clean_10s_pcm16_sr16000.wav"), ecg[:fs*10], 16000)
sf.write(os.path.join(base_path, "100_ytstyle_noisy_10s_pcm16_sr16000.wav"), noisy_ecg[:fs*10], 16000)
sf.write(os.path.join(base_path, "100_ytstyle_filtered_10s_pcm16_sr16000.wav"), ecg_filtered[:fs*10], 16000)

print("✅ Saved 10-second clean, noisy, and filtered ECG audio files.")


In [ ]:
# --- ECG Audio Buttons: Clean, Noisy, Filtered, Demo ---
import os
from IPython.display import display
import ipywidgets as widgets

BASE = r"C:\notch_filter_pocs\notch_project\data_ecg\mit-bih-arrhythmia-database-1.0.0\mit-bih-arrhythmia-database-1.0.0"

FILES = {
    "🎧 Play CLEAN ECG (original)": os.path.join(BASE, "100_clean_10s_pcm16_sr16000.wav"),
    "⚡ Play NOISY ECG (with hum + interference)": os.path.join(BASE, "100_ytstyle_noisy_10s_pcm16_sr16000.wav"),
    "💚 Play FILTERED ECG (after notch + bandpass)": os.path.join(BASE, "100_ytstyle_filtered_10s_pcm16_sr16000.wav")
}

out = widgets.Output()

def make_handler(path):
    def _handler(_):
        with out:
            print(f"\n▶️  Opening:", os.path.basename(path))
        try:
            os.startfile(path)
        except Exception as e:
            with out:
                print("⚠️  Could not open via startfile:", e)
    return _handler

buttons = []
for label, path in FILES.items():
    btn = widgets.Button(
        description=label,
        layout=widgets.Layout(width='420px', height='40px'),
        style={'button_color': '#1f77b4', 'font_weight': 'bold'}
    )
    btn.on_click(make_handler(path))
    buttons.append(btn)

print("🎵 Click any button below to play the corresponding ECG signal in your system player:\n")
display(*buttons)
display(out)


In [ ]:
# 🎛️ Final button panel — plays Clean / Noisy / Filtered using simpleaudio (reliable)
# Install simpleaudio if you haven't: pip install simpleaudio
import os
from IPython.display import display
import ipywidgets as widgets

# Try importing simpleaudio
try:
    import simpleaudio as sa
    SIMPLEAUDIO_AVAILABLE = True
except Exception as e:
    print("simpleaudio not available — buttons will fallback to opening the file externally.")
    SIMPLEAUDIO_AVAILABLE = False

BASE = r"C:\notch_filter_pocs\notch_project\data_ecg\mit-bih-arrhythmia-database-1.0.0\mit-bih-arrhythmia-database-1.0.0"

FILES = {
    "🎧 Play CLEAN ECG (original)": os.path.join(BASE, "100_clean_10s_pcm16_sr16000.wav"),
    "⚡ Play NOISY ECG (with hum + interference)": os.path.join(BASE, "100_ytstyle_noisy_10s_pcm16_sr16000.wav"),
    "💚 Play FILTERED ECG (after notch + bandpass)": os.path.join(BASE, "100_ytstyle_filtered_10s_pcm16_sr16000.wav"),
    "🩺 Play DEMO (clean → noisy → filtered)": os.path.join(BASE, "100_demo_trio_sr16000.wav"),
}

out = widgets.Output()
display(widgets.HTML("<b>🎵 Click a button to play. Press another button to stop current playback and play the new one.</b><br><br>"))

# keep reference to current PlayObject so we can stop it
_current_play = {"obj": None}

def stop_current():
    po = _current_play.get("obj")
    try:
        if po is not None:
            po.stop()
    except Exception:
        pass
    _current_play["obj"] = None

def make_handler(path):
    def _handler(_):
        with out:
            print(f"\n▶️  Requested: {os.path.basename(path)}")
        # stop anything playing now
        stop_current()
        if SIMPLEAUDIO_AVAILABLE:
            try:
                # load and play asynchronously (non-blocking)
                wave_obj = sa.WaveObject.from_wave_file(path)
                play_obj = wave_obj.play()
                _current_play["obj"] = play_obj
                with out:
                    print("Playing via simpleaudio.")
                return
            except Exception as e:
                with out:
                    print("simpleaudio playback failed:", e)
        # fallback: open externally (Windows Media Player)
        try:
            with out:
                print("Falling back to external player (os.startfile).")
            os.startfile(path)
        except Exception as e:
            with out:
                print("External open failed:", e)
    return _handler

# Build buttons
buttons = []
for label, path in FILES.items():
    btn = widgets.Button(
        description=label,
        layout=widgets.Layout(width='520px', height='42px'),
        style={'button_color': '#1f77b4', 'font_weight': 'bold'}
    )
    btn.on_click(make_handler(path))
    buttons.append(btn)

# Stop button
stop_btn = widgets.Button(description="⏹ Stop Playback", layout=widgets.Layout(width='200px'))
def _stop_click(b):
    stop_current()
    with out:
        print("Stopped playback.")
stop_btn.on_click(_stop_click)

# Display UI
control_row = widgets.HBox([stop_btn])
display(*buttons)
display(control_row)
display(out)


In [ ]:
# 🎛️ ECG Audio Buttons — Windows-only, no installs needed
# Plays via winsound using a short temp file path; includes Stop button.
import os, shutil, tempfile, importlib, winsound
from IPython.display import display
import ipywidgets as widgets

# Reload winsound to avoid missing-constant weirdness in Jupyter
importlib.reload(winsound)

BASE = r"C:\notch_filter_pocs\notch_project\data_ecg\mit-bih-arrhythmia-database-1.0.0\mit-bih-arrhythmia-database-1.0.0"

FILES = {
    "🎧 Play CLEAN ECG (original)": os.path.join(BASE, "100_clean_10s_pcm16_sr16000.wav"),
    "⚡ Play NOISY ECG (with hum + interference)": os.path.join(BASE, "100_ytstyle_noisy_10s_pcm16_sr16000.wav"),
    "💚 Play FILTERED ECG (after notch + bandpass)": os.path.join(BASE, "100_ytstyle_filtered_10s_pcm16_sr16000.wav"),
    "🩺 Play DEMO (clean → noisy → filtered)": os.path.join(BASE, "100_demo_trio_sr16000.wav"),
}

# Short temp path to avoid Windows MCI long-path headaches
TMP_DIR = tempfile.gettempdir()
TMP_WAV = os.path.join(TMP_DIR, "ecg_play.wav")  # e.g., C:\Users\<you>\AppData\Local\Temp\ecg_play.wav

# Helper: stop any current playback
def stop_playback():
    try:
        # 0 means SND_PURGE to stop; using PlaySound(None, 0) also stops current sound
        winsound.PlaySound(None, 0)
    except Exception:
        pass

# Handler factory
def make_handler(src_path):
    def _handler(_):
        with out:
            print(f"\n▶️  {os.path.basename(src_path)}")
        try:
            # Stop anything that might be playing
            stop_playback()
            # Copy to short temp path
            shutil.copy(src_path, TMP_WAV)
            # 1 = SND_FILENAME, 0x0001 = SND_ASYNC (play without blocking)
            winsound.PlaySound(TMP_WAV, 1 | 0x0001)
            with out:
                print(f"Playing via winsound from: {TMP_WAV}")
        except Exception as e:
            with out:
                print("⚠️ Playback error, opening externally:", e)
            try:
                os.startfile(src_path)  # fallback external player
            except Exception as e2:
                with out:
                    print("External open failed:", e2)
    return _handler

# Build UI
buttons = []
for label, path in FILES.items():
    btn = widgets.Button(
        description=label,
        layout=widgets.Layout(width='520px', height='42px'),
        style={'button_color': '#1f77b4', 'font_weight': 'bold'}
    )
    btn.on_click(make_handler(path))
    buttons.append(btn)

stop_btn = widgets.Button(description="⏹ Stop Playback", layout=widgets.Layout(width='200px'))
def _stop(_):
    stop_playback()
    with out:
        print("Stopped playback.")
stop_btn.on_click(_stop)

out = widgets.Output()
display(widgets.HTML("🎵 <b>Click a button to play. Press another button or 'Stop' to stop current playback.</b><br><br>"))
display(*buttons)
display(widgets.HBox([stop_btn]))
display(out)


In [ ]:
# Spectrogram comparison (clean, noisy, filtered) — 5 s window for clarity
from scipy import signal
import matplotlib.pyplot as plt
import numpy as np

# Use first 5 s (or available length)
nsec = 5
nsamps = min(len(ecg), int(nsec * fs))
clip_clean = ecg[:nsamps]
clip_noisy = noisy_ecg[:nsamps]
clip_filt = ecg_filtered[:nsamps]

def plot_spectrogram(x, fs, ax, title):
    f, t_spec, Sxx = signal.spectrogram(x, fs, nperseg=512, noverlap=384)
    Sxx_db = 10 * np.log10(Sxx + 1e-12)
    im = ax.pcolormesh(t_spec, f, Sxx_db, shading='gouraud', cmap='viridis')
    ax.set_ylim(0, 200)
    ax.set_ylabel('Freq (Hz)')
    ax.set_xlabel('Time (s)')
    ax.set_title(title)
    return im

plt.figure(figsize=(12,9))
ax1 = plt.subplot(3,1,1)
im1 = plot_spectrogram(clip_clean, fs, ax1, 'Spectrogram — Clean ECG')
ax2 = plt.subplot(3,1,2)
im2 = plot_spectrogram(clip_noisy, fs, ax2, 'Spectrogram — Noisy ECG (50 Hz hum visible)')
ax3 = plt.subplot(3,1,3)
im3 = plot_spectrogram(clip_filt, fs, ax3, 'Spectrogram — Filtered ECG (after notch + bandpass)')
plt.tight_layout()
cbar = plt.colorbar(im3, ax=[ax1,ax2,ax3], orientation='vertical', fraction=0.02, pad=0.02)
cbar.set_label('Power (dB)')
plt.show()


In [ ]:
# PSD comparison using Welch's method (shows 50 Hz removal)
from scipy import signal
import matplotlib.pyplot as plt
import numpy as np

# use 10 s (or available)
nsec = 10
nsamps = min(len(ecg), int(nsec*fs))
clean_seg = ecg[:nsamps]
noisy_seg = noisy_ecg[:nsamps]
filt_seg = ecg_filtered[:nsamps]

f_c, P_c = signal.welch(clean_seg, fs, nperseg=2048)
f_n, P_n = signal.welch(noisy_seg, fs, nperseg=2048)
f_f, P_f = signal.welch(filt_seg, fs, nperseg=2048)

plt.figure(figsize=(10,5))
plt.semilogy(f_n, P_n, label='Noisy', color='orange', alpha=0.9)
plt.semilogy(f_f, P_f, label='Filtered', color='royalblue', alpha=0.9)
plt.semilogy(f_c, P_c, label='Clean', color='green', alpha=0.7)
plt.xlim(0, 120)
plt.xlabel('Frequency (Hz)')
plt.ylabel('PSD')
plt.title('PSD Comparison — check 50 Hz notch removal')
plt.axvline(50, color='red', linestyle='--', linewidth=1, label='50 Hz')
# annotate SNR improvements if available (you computed earlier)
try:
    snr_noisy = compute_snr(clean_seg, noisy_seg)
    snr_filt  = compute_snr(clean_seg, filt_seg)
    plt.text(60, max(P_n)*0.5, f"SNR noisy: {snr_noisy:.2f} dB\nSNR filtered: {snr_filt:.2f} dB")
except Exception:
    pass
plt.legend()
plt.grid(which='both', linestyle='--', linewidth=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# Save 10 s clips (PCM16, 16k) and write a small summary file
import soundfile as sf
import os, datetime

out_base = base_path  # same folder as earlier
sr_out = 16000

# helper to resample and write final 16k files (if needed)
def ensure_and_write(name, arr, src_sr):
    outname = os.path.join(out_base, name)
    # if arr sr equals sr_out, write directly, else resample
    if src_sr != sr_out:
        from scipy import signal
        import math
        up = sr_out; down = src_sr
        g = math.gcd(up, down); up//=g; down//=g
        y = signal.resample_poly(arr, up, down)
    else:
        y = arr
    # normalize a touch
    if y.max() != 0:
        y = y / (np.max(np.abs(y)) + 1e-12) * 0.95
    sf.write(outname, y.astype('float32'), sr_out, subtype='PCM_16')
    return outname

# 10-second slices (or full if shorter)
L = min(len(ecg), sr_out*10)
clean10 = ecg[:int(10*fs)] if fs==sr_out else ecg[:int(10*fs)]
noisy10 = noisy_ecg[:int(10*fs)]
filt10  = ecg_filtered[:int(10*fs)]

# Write (these will produce files in your base_path)
fn_clean = ensure_and_write("100_clean_final_10s_sr16k.wav", clean10, fs)
fn_noisy = ensure_and_write("100_noisy_final_10s_sr16k.wav", noisy10, fs)
fn_filt  = ensure_and_write("100_filtered_final_10s_sr16k.wav", filt10, fs)

# write summary text
summary_path = os.path.join(out_base, "results_summary.txt")
with open(summary_path, "w") as f:
    f.write("ECG Denoising Results Summary\n")
    f.write(f"Timestamp: {datetime.datetime.now()}\n")
    f.write(f"Record: 100\n")
    f.write(f"Sampling rate (original): {fs} Hz\n")
    f.write(f"Saved files:\n - {os.path.basename(fn_clean)}\n - {os.path.basename(fn_noisy)}\n - {os.path.basename(fn_filt)}\n")
    try:
        f.write(f"SNR noisy: {snr_noisy:.2f} dB\nSNR filtered: {snr_filt:.2f} dB\n")
    except Exception:
        pass
    f.write("\nFilters used: 50 Hz IIR notch (Q=30), 4th-order Butterworth bandpass 0.5–45 Hz\n")
print("Saved files and summary to:", out_base)
print("Summary file:", summary_path)


In [ ]:
# 🔥 Single combined figure: Waveform (overlay) + Spectrograms (noisy vs filtered) + PSD
import os
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal

# use these variables from your notebook: ecg, noisy_ecg, ecg_filtered, fs, base_path
# if base_path not defined, set it to the folder where you want the PNG saved
try:
    base_path
except NameError:
    base_path = r"C:\notch_filter_pocs\notch_project\data_ecg\mit-bih-arrhythmia-database-1.0.0\mit-bih-arrhythmia-database-1.0.0"

# Parameters for plots
wave_time_s = 4        # seconds to show in waveform plot
spec_time_s = 6        # seconds used for spectrograms
psd_time_s = 10        # seconds used for PSD

# prepare segments
n_w = min(len(ecg), int(wave_time_s * fs))
n_s = min(len(ecg), int(spec_time_s * fs))
n_p = min(len(ecg), int(psd_time_s * fs))

t_w = np.arange(n_w) / fs

wave_clean = ecg[:n_w]
wave_noisy = noisy_ecg[:n_w]
wave_filt  = ecg_filtered[:n_w]

spec_clean = ecg[:n_s]
spec_noisy = noisy_ecg[:n_s]
spec_filt  = ecg_filtered[:n_s]

psd_clean = ecg[:n_p]
psd_noisy = noisy_ecg[:n_p]
psd_filt  = ecg_filtered[:n_p]

# make figure
fig = plt.figure(figsize=(12,12))
gs = fig.add_gridspec(4, 2, height_ratios=[1.2, 1.8, 1.8, 1.2], hspace=0.45)

# Row 1: waveform overlay (span both columns)
ax_wave = fig.add_subplot(gs[0, :])
ax_wave.plot(t_w, wave_clean, label='Clean', color='g', linewidth=1.2)
ax_wave.plot(t_w, wave_noisy, label='Noisy', color='tab:orange', alpha=0.5)
ax_wave.plot(t_w, wave_filt,  label='Filtered', color='b', alpha=0.9)
ax_wave.set_xlim(0, wave_time_s)
ax_wave.set_ylabel('Amplitude')
ax_wave.set_title('Waveform (first {:.0f} s)'.format(wave_time_s))
ax_wave.legend(loc='upper right')
ax_wave.grid(alpha=0.25)

# Row 2: spectrogram noisy (left), spectrogram filtered (right)
def draw_spec(ax, x, sr, title):
    f, t_spec, Sxx = signal.spectrogram(x, sr, nperseg=512, noverlap=384)
    Sxx_db = 10 * np.log10(Sxx + 1e-12)
    im = ax.pcolormesh(t_spec, f, Sxx_db, shading='gouraud', cmap='viridis')
    ax.set_ylim(0, 200)
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Freq (Hz)')
    ax.set_title(title)
    return im

ax_spec_n = fig.add_subplot(gs[1, 0])
im1 = draw_spec(ax_spec_n, spec_noisy, fs, 'Spectrogram — Noisy (50 Hz visible)')
ax_spec_f = fig.add_subplot(gs[1, 1])
im2 = draw_spec(ax_spec_f, spec_filt, fs, 'Spectrogram — Filtered (50 Hz removed)')

# colorbar for spectrograms (use the last image's scale)
cax = fig.add_axes([0.92, 0.55, 0.015, 0.25])  # left, bottom, width, height (normalized)
plt.colorbar(im2, cax=cax, label='Power (dB)')

# Row 3: PSD (span both columns)
ax_psd = fig.add_subplot(gs[2:, :])
f_c, P_c = signal.welch(psd_clean, fs, nperseg=2048)
f_n, P_n = signal.welch(psd_noisy, fs, nperseg=2048)
f_f, P_f = signal.welch(psd_filt, fs, nperseg=2048)

ax_psd.semilogy(f_n, P_n, label='Noisy', color='tab:orange')
ax_psd.semilogy(f_f, P_f, label='Filtered', color='tab:blue')
ax_psd.semilogy(f_c, P_c, label='Clean', color='green', alpha=0.7)
ax_psd.set_xlim(0, 120)
ax_psd.set_xlabel('Frequency (Hz)')
ax_psd.set_ylabel('PSD')
ax_psd.set_title('PSD Comparison — 50 Hz notch removal')
ax_psd.axvline(50, color='red', linestyle='--', linewidth=1)
ax_psd.text(52, max(np.max(P_n), np.max(P_f))/5, '50 Hz', color='red')
ax_psd.legend()
ax_psd.grid(which='both', linestyle='--', alpha=0.3)

# tidy & save
plt.tight_layout(rect=[0,0,0.9,1])  # leave space for vertical colorbar
out_png = os.path.join(base_path, 'ecg_final_figure.png')
plt.savefig(out_png, dpi=200)
plt.show()
print("Saved combined figure to:", out_png)
